In [15]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio]
!pip install git+https://github.com/huggingface/transformers.git@main


  Cloning https://github.com/huggingface/transformers.git (to revision main) to /tmp/pip-req-build-17jet42r
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-17jet42r
ERROR: Operation cancelled by user


In [ ]:
import IPython
from datasets import load_dataset, Audio
from transformers import MimiModel, AutoFeatureExtractor

The following codeblock is to get started with the Mimi model using a dummy example from the LibriSpeech dataset (~9MB).
Source: https://huggingface.co/kyutai/mimi

In [ ]:
# load a demonstration datasets
librispeech_dummy = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

In [21]:
# load the model + feature extractor (for pre-processing the audio)
model = MimiModel.from_pretrained("kyutai/mimi")
feature_extractor = AutoFeatureExtractor.from_pretrained("kyutai/mimi")

Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]

In [ ]:
# cast the audio data to the correct sampling rate for the model
librispeech_dummy = librispeech_dummy.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))
sample = librispeech_dummy[3]
audio_sample = sample["audio"]
print(sample["text"])
IPython.display.Audio(audio_sample["array"], rate=audio_sample["sampling_rate"])

HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA


In [26]:
# pre-process the inputs
inputs = feature_extractor(raw_audio=audio_sample["array"], sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
inputs

{'padding_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32), 'input_values': tensor([[[-0.0005, -0.0004, -0.0004,  ..., -0.0002, -0.0002, -0.0003]]])}

In [45]:
# explicitly encode then decode the audio inputs / for mimi-tokens
encoder_outputs = model.encode(inputs["input_values"])
encoder_outputs.audio_codes.shape[1]

32

In [ ]:
# explicitly encode then decode the audio inputs / for audio-tokens
audio_values = model.decode(encoder_outputs.audio_codes)[0]
audio_decoded = audio_values.squeeze().detach().cpu().numpy()
IPython.display.Audio(audio_decoded, rate=feature_extractor.sampling_rate)
# or the equivalent with a forward pass
# audio_values = model(inputs["input_values"]).audio_values